# Install the required libraries

In [1]:
pip install requests beautifulsoup4 pandas lxml

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime

# Define RSS Sources

In [3]:
RSS_SOURCES = {
    "Global LNG Hub":     "https://globallnghub.com/feed",
    "LNG Journal":        "https://lngjournal.com/index.php/feed",
    "NaturalGasWorld":    "https://www.naturalgasworld.com/rss",
    "Offshore Technology":"https://www.offshore-technology.com/feed/",
    "S&P Global LNG":     "https://www.spglobal.com/commodityinsights/en/rss-feed/natural-gas",
    "EIA":                "https://www.eia.gov/rss/press_rss.xml",
}


# HTML scrape for Straits Times and Business Times

In [4]:
HTML_SOURCES = {
    "Business Times": {
        "url":            "https://www.businesstimes.com.sg/keywords/liquefied-natural-gas",
        "headline_tag":   "h3",
        "headline_class": "story-card__heading",
        "date_tag":       "time",
        "date_class":     None,
    },
    "Straits Times": {
        "url":            "https://www.straitstimes.com/tags/lng",
        "headline_tag":   "h5",
        "headline_class": "card-title",
        "date_tag":       "time",
        "date_class":     None,
    },
}

# Keywords to Filter LNG-relevant headlines

In [5]:
LNG_KEYWORDS = ["LNG", "liquefied natural gas", "natural gas", "regasification",
                "JKM", "TTF", "Henry Hub", "gas price", "bunkering",
                "Singapore LNG", "SLNG", "reload", "cargo", "GasCo",
                "Hormuz", "Qatar", "energy market"]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0.0.0 Safari/537.36"
}

In [6]:
#Helper Functions
def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'[^\w\s\-\.,]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def parse_date(raw_date):
    if not raw_date:
        return datetime.today().strftime("%Y-%m-%d")
    formats = [
        "%a, %d %b %Y %H:%M:%S %z",
        "%a, %d %b %Y %H:%M:%S GMT",
        "%Y-%m-%dT%H:%M:%S%z",
        "%Y-%m-%d",
    ]
    for fmt in formats:
        try:
            return datetime.strptime(raw_date.strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return raw_date

def is_lng_relevant(headline):
    return any(kw.lower() in headline.lower() for kw in LNG_KEYWORDS)

def extract_price(text):
    match = re.search(r'\$?\d+\.?\d*\s*(\/MMBtu|USD|per\s+tonne)?', text)
    return match.group(0) if match else ""

all_headlines = []

# Scrape all RSS feeds

In [7]:
for source_name, url in RSS_SOURCES.items():
    print(f"Fetching RSS: {source_name} ...")
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.content, "lxml-xml")
        items = soup.find_all("item")
        print(f"  Found {len(items)} items")

        for item in items:
            headline = clean_text(item.find("title").get_text() if item.find("title") else "")
            pub_date = parse_date(item.find("pubDate").get_text() if item.find("pubDate") else "")
            link     = item.find("link").get_text() if item.find("link") else ""

            if is_lng_relevant(headline):
                all_headlines.append({
                    "date":            pub_date,
                    "headline":        headline,
                    "source":          source_name,
                    "url":             link.strip(),
                    "price_mentioned": extract_price(headline)
                })

    except Exception as e:
        print(f"  SKIPPED {source_name}: {e}")

Fetching RSS: Global LNG Hub ...
  Found 10 items
Fetching RSS: LNG Journal ...
  Found 0 items
Fetching RSS: NaturalGasWorld ...
  Found 20 items
Fetching RSS: Offshore Technology ...
  Found 0 items
Fetching RSS: S&P Global LNG ...
  Found 0 items
Fetching RSS: EIA ...
  Found 7 items


In [8]:
!pip install feedparser

In [9]:
import feedparser

ST_FEEDS = {
    "Straits Times Business": "https://www.straitstimes.com/rss/business",
    "Straits Times Singapore": "https://www.straitstimes.com/rss/singapore",
}

for source_name, url in ST_FEEDS.items():
    feed = feedparser.parse(url)
    print(f"{source_name}: {len(feed.entries)} entries")
    for entry in feed.entries:
        headline = clean_text(entry.get("title", ""))
        pub_date = parse_date(entry.get("published", ""))
        if is_lng_relevant(headline):
            all_headlines.append({
                "date": pub_date, "headline": headline,
                "source": source_name, "url": entry.get("link", ""),
                "price_mentioned": extract_price(headline)
            })

Straits Times Business: 0 entries
Straits Times Singapore: 0 entries


# Scrape HTML pages (ST and BT)

In [10]:
for source_name, config in HTML_SOURCES.items():
    print(f"Fetching HTML: {source_name} ...")
    try:
        response = requests.get(config["url"], headers=HEADERS, timeout=15)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, "html.parser")

        # Find headline elements
        if config["headline_class"]:
            tags = soup.find_all(config["headline_tag"], class_=config["headline_class"])
        else:
            tags = soup.find_all(config["headline_tag"])

        # If class-based search fails, fall back to all article links
        if not tags:
            tags = soup.find_all("a", href=True)

        print(f"  Found {len(tags)} candidate elements")

        for tag in tags:
            headline = clean_text(tag.get_text())
            # Try to find a nearby date
            parent = tag.find_parent()
            time_tag = parent.find("time") if parent else None
            pub_date = parse_date(
                time_tag.get("datetime", "") if time_tag else ""
            )

            if headline and len(headline) > 15 and is_lng_relevant(headline):
                # Try to extract link
                a_tag = tag.find("a") or (tag if tag.name == "a" else None)
                link = ""
                if a_tag and a_tag.get("href"):
                    href = a_tag["href"]
                    link = href if href.startswith("http") else f"https://www.{source_name.lower().replace(' ','')}.com.sg{href}"

                all_headlines.append({
                    "date":            pub_date,
                    "headline":        headline,
                    "source":          source_name,
                    "url":             link,
                    "price_mentioned": extract_price(headline)
                })

    except Exception as e:
        print(f"  SKIPPED {source_name}: {e}")

Fetching HTML: Business Times ...
  Found 151 candidate elements
Fetching HTML: Straits Times ...
  Found 2 candidate elements


# Save to CSV

In [11]:
import os  # Import os module to create directories

df = pd.DataFrame(all_headlines)
df = df.drop_duplicates(subset=["headline"])
df = df[df["headline"].str.len() > 15]   # Remove stub/empty headlines
df = df.sort_values("date", ascending=False).reset_index(drop=True)

output_path = "data/raw/headlines_raw.csv"

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df.to_csv(output_path, index=False, encoding='utf-8')

print(f"\n✅ Saved {len(df)} LNG headlines to {output_path}")
print(f"   Sources breakdown:\n{df['source'].value_counts()}")
print(df[['date','headline','source']].head(10).to_string())


✅ Saved 30 LNG headlines to data/raw/headlines_raw.csv
   Sources breakdown:
source
NaturalGasWorld    11
Global LNG Hub     10
Business Times      7
EIA                 2
Name: count, dtype: int64
                            date                                                                                             headline          source
0   Tue, 7 Apr 2026 12:00:00 EST                Hormuz closure and related production outages are key drivers in EIAs latest forecast             EIA
1  Tue, 10 Feb 2026 12:00:00 EST  EIA raises natural gas price forecast following increased heating demand amid severe winter weather             EIA
2                     2026-06-04                                      Vietnam buys more LNG as temperatures set to rise above average  Business Times
3                     2026-06-04                                            First LNG shipment since war began appears to exit Hormuz  Business Times
4                     2026-06-04                   

In [12]:
# Check saved file
df = pd.read_csv("data/raw/headlines_raw.csv", encoding='utf-8')
print(f"Loaded {len(df)} headlines")
df.head(3)

Loaded 30 headlines


,date,headline,source,url,price_mentioned
0,"Tue, 7 Apr 2026 12:00:00 EST",Hormuz closure and related production outages ...,EIA,/pressroom/releases/press586.php,NaN
1,"Tue, 10 Feb 2026 12:00:00 EST",EIA raises natural gas price forecast followin...,EIA,/pressroom/releases/press583.php,NaN
2,2026-06-04,Vietnam buys more LNG as temperatures set to r...,Business Times,https://www.businesstimes.com.sg/international...,NaN


In [13]:
# Apply all cleaning and extraction to headlines_raw.csv

from html import unescape

# Define the missing functions
def strip_html_tags(text):
    """Remove HTML tags from text"""
    if pd.isna(text):
        return text
    return re.sub(r'<[^>]+>', '', str(text))

def remove_html_entities(text):
    """Remove HTML entities like &amp;, &lt;, etc."""
    if pd.isna(text):
        return text
    return unescape(str(text))

def remove_special_chars(text):
    """Remove special characters, keeping only alphanumeric and basic punctuation"""
    if pd.isna(text):
        return text
    return re.sub(r'[^\w\s\.\,\!\?\-\$\%]', '', str(text))

def extract_dates(text):
    """Extract dates from text - returns list of dictionaries"""
    if pd.isna(text):
        return []
    # Simple date pattern matching (you can expand this)
    date_pattern = r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b'
    matches = re.findall(date_pattern, str(text))
    return [{"raw_date": match} for match in matches]

def extract_lng_prices(text):
    """Extract prices from text - returns list of dictionaries"""
    if pd.isna(text):
        return []
    # Pattern to match prices like $123.45, £50, €100, etc.
    price_pattern = r'[\$£€¥]\s*\d+(?:[\.,]\d{2})?|\d+(?:[\.,]\d{2})?\s*(?:dollars?|pounds?|euros?|USD|GBP|EUR)'
    matches = re.findall(price_pattern, str(text), re.IGNORECASE)
    return [{"price_value": match} for match in matches]

# Apply all cleaning and extraction to headlines_raw.csv
df["headline_clean"] = (
    df["headline"]
    .apply(strip_html_tags)
    .apply(remove_html_entities)
    .apply(remove_special_chars)
)

# Extract first date found in headline (if any)
df["date_extracted"] = df["headline_clean"].apply(
    lambda x: extract_dates(x)[0]["raw_date"] if extract_dates(x) else None
)

# Extract first price found in headline (if any) - Added error handling
df["price_extracted"] = df["headline_clean"].apply(
    lambda x: extract_lng_prices(x)[0]["price_value"] if extract_lng_prices(x) else None
)

# Use existing date column if date_extracted is empty
df["date_final"] = df["date_extracted"].fillna(df["date"])

# Save cleaned version
df.to_csv("data/raw/headlines_clean.csv", index=False, encoding='utf-8')

print(f"✅ Saved cleaned dataset with {len(df)} rows")
print(f"   Headlines with prices extracted: {df['price_extracted'].notna().sum()}")
print(f"   Headlines with dates extracted:  {df['date_extracted'].notna().sum()}")
df[['date_final','headline_clean','price_extracted','source']].head(10)

✅ Saved cleaned dataset with 30 rows
   Headlines with prices extracted: 0
   Headlines with dates extracted:  0


,date_final,headline_clean,price_extracted,source
0,"Tue, 7 Apr 2026 12:00:00 EST",Hormuz closure and related production outages ...,None,EIA
1,"Tue, 10 Feb 2026 12:00:00 EST",EIA raises natural gas price forecast followin...,None,EIA
2,2026-06-04,Vietnam buys more LNG as temperatures set to r...,None,Business Times
3,2026-06-04,First LNG shipment since war began appears to ...,None,Business Times
4,2026-06-04,Australia requires LNG exporters to reserve 20...,None,Business Times
5,2026-06-04,China LNG imports hint at recovery as buyers r...,None,Business Times
6,2026-06-04,Why US70 should be the most worrying number fo...,None,Business Times
7,2026-06-04,Singapore GasCo has bought enough LNG to last ...,None,Business Times
8,2026-06-04,A scorching Asian summer will add to risk of s...,None,Business Times
9,2026-06-03,"Natural gas prices weekly update JKM, TTF and ...",None,Global LNG Hub


# Data Provenance, Quality & Limitations

**Project:** Singapore LNG Market Intelligence Engine  
**Author:** Priya Thanarajan  
**Date:** June 2026  
**Notebook:** `01_data_sourcing.ipynb`

## Purpose of This Document

This cell records the origin, quality, and known limitations of every dataset used in this project. It demonstrates understanding of how data was generated, where it comes from, and what constraints apply to its use in modelling.

## Data Sources

| # | Dataset | Description | Source | URL | Format | Date Range | Frequency |
|---|---|---|---|---|---|---|---|
| 1 | Asia LNG Spot Price (JKM) | Japan-Korea Marker — benchmark Asia LNG spot price in USD/MMBtu | World Bank / FRED | [FRED: PNGASJPUSDM](https://fred.stlouisfed.org/series/PNGASJPUSDM) | CSV | Jan 1992 – Mar 2026 | Monthly |
| 2 | Henry Hub Natural Gas Price | US benchmark gas price in USD/MMBtu | FRED | [FRED: DHHNGSP](https://fred.stlouisfed.org/series/DHHNGSP) | CSV | Jan 1997 – May 2026 | Daily |
| 3 | TTF European Gas Price | European benchmark gas price in USD/MMbtu | World Bank / FRED| [FRED: PNGASEUUSDM](https://fred.stlouisfed.org/series/PNGASEUUSDM#) | CSV | Jan 1992 – Mar 2026 | Monthly |
| 4 | LNG-T3 Global Trade Flows | AIS-derived daily LNG terminal throughput and country-to-country trade flows | Zenodo (Open Access) | [zenodo.org/records/17273527](https://zenodo.org/records/17273527) | CSV (5 files) | Jan 2020 – Dec 2024 | Daily |
| 5 | Singapore USEP Electricity Price | Average monthly Uniform Singapore Energy Price in SGD/MWh | Energy Market Authority (EMA) | [ema.gov.sg/resources/statistics](https://www.ema.gov.sg/resources/singapore-energy-statistics) | CSV | Jan 2005 – Present | Monthly |
| 6 | LNG News Headlines | Scraped headlines from LNG trade and energy news sources | Web Scraping (BeautifulSoup) | See sources below | CSV | 2020 – 2026 | Daily (scraped) |


TTF European Gas Price	FRED (IMF Primary Commodity Prices)	
FRED: PNGASEUUSDM
USD/MMBtu	Jan 1992 – Mar 2026

### Headline Scraping Sources

| Source | URL | Type |
|---|---|---|
| Global LNG Hub | https://globallnghub.com/feed | RSS |
| LNG Journal | https://lngjournal.com/index.php/feed | RSS |
| NaturalGasWorld | https://www.naturalgasworld.com/rss | RSS |
| S&P Global Commodity Insights | https://www.spglobal.com/commodityinsights | RSS |
| Business Times Singapore | https://www.businesstimes.com.sg/keywords/liquefied-natural-gas | HTML scrape |
| Straits Times Singapore | https://www.straitstimes.com/tags/lng | HTML scrape |

## How Each Dataset Was Generated

### 1. JKM Asia LNG Spot Price (FRED)
The Japan-Korea Marker (JKM) is a spot price assessment published by **Platts (S&P Global)**, reflecting the market price for LNG cargoes delivered to Japan or Korea. It is the most widely used benchmark for Asian LNG trading. The FRED series aggregates these monthly assessments. The price reflects free-on-board (FOB) cargo trades and spot tender results — it is an **assessed price**, not a transaction price, meaning it is based on market intelligence rather than recorded transactions directly.

### 2. Henry Hub Price (FRED)
Published daily by the **US Energy Information Administration (EIA)** based on natural gas traded at the Henry Hub pipeline interconnection in Louisiana. This is a direct transaction-based price from the NYMEX futures market, making it more transparent than assessed prices.

### 3. TTF European Gas Price (World Bank)
The Title Transfer Facility (TTF) is the Dutch virtual gas trading hub price, administered by **Gasunie Transport Services**. It is the dominant European gas benchmark and increasingly used as a reference for global LNG arbitrage calculations.

### 4. LNG-T3 Dataset (Zenodo)
Constructed from **Automatic Identification System (AIS)** vessel position data — the GPS-equivalent signal broadcast by all commercial ships. The dataset infers terminal activity and cargo volumes from vessel draught changes (deeper draft = loaded with LNG). It covers 406 LNG tankers and 545 terminals globally. Validation was performed against Gas Infrastructure Europe (GIE), EIA, Eurostat, and GIIGNL annual reports.

### 5. Singapore USEP (EMA)
The Uniform Singapore Energy Price is the half-hourly wholesale electricity price in Singapore's National Electricity Market (NEMS), administered by the **Energy Market Authority**. It reflects the cost of gas-fired power generation and is directly influenced by LNG import prices.

### 6. News Headlines (Web Scraped)
Headlines were collected via HTTP requests to RSS feeds and keyword-tag pages. Text was cleaned using Regex (HTML tag stripping, entity removal, special character removal). Dates were parsed from RSS `<pubDate>` fields. Sentiment labels were not manually verified — they rely on automated NLP classification.


## Data Quality Assessment

| Dataset | Completeness | Accuracy | Consistency | Known Issues |
|---|---|---|---|---|
| JKM (FRED) | ✅ High — monthly back to 1992 | ✅ Assessed by Platts, industry standard | ✅ Consistent methodology | Price is assessed, not directly transacted |
| Henry Hub (FRED) | ✅ High — daily back to 1997 | ✅ Transaction-based, NYMEX | ✅ Consistent | Some bank holidays show no price |
| TTF (World Bank) | ✅ High — daily back to 1997  | ✅ Exchange-based | ⚠️ Unit changes (EUR/MWh vs $/MMBtu) | Requires unit conversion for comparison |
| LNG-T3 (Zenodo) | ⚠️ Medium — AIS coverage varies | ⚠️ Estimated volumes (draught-based) | ✅ Daily, consistent 2020–2024 | Volume estimation error ±10-15%; some vessels not tracked |
| USEP (EMA) | ✅ High — official government data | ✅ Regulatory source | ✅ Consistent | Only covers Singapore domestic market |
| Headlines (Scraped) | ⚠️ Medium — dependent on feed availability | ⚠️ No fact-checking applied | ⚠️ Variable by source | Duplicate headlines possible; paywalled articles not captured |

## Known Limitations

### 1. JKM Price is Assessed, Not Transacted
The JKM price is a **market assessment** published by Platts, not a record of actual transaction prices. This means it reflects expert judgment about where the market is trading rather than direct trade confirmation. This introduces a degree of subjectivity and potential lag.

### 2. LNG-T3 Volume Estimates Have Uncertainty
Cargo volumes in LNG-T3 are inferred from **vessel draught changes** (how deep the ship sits in the water), not from direct metering. The dataset authors estimate a volume accuracy of approximately ±10–15%. This means aggregate trade flow figures should be interpreted as indicative rather than precise.

### 3. Headline Data is Incomplete and Biased
Web-scraped headlines represent only what is publicly published on RSS feeds and keyword pages. **Paywalled articles** (e.g., full Straits Times or Business Times content) are not captured. This introduces a **selection bias** — only freely available news is included, which may over-represent certain sources or topics.

### 4. LNG-T3 Ends December 2024
The LNG-T3 dataset covers only up to December 2024. For 2025–2026 trade flow features, proxy variables (e.g., EIA US LNG export volumes, GIIGNL summary data) will be used to fill the gap in the modelling pipeline.

### 5. Regulatory and Market Regime Changes
Singapore's LNG market underwent significant changes in 2022 (post-Ukraine energy crisis) and again in early 2026 (Strait of Hormuz closure, Qatar force majeure). These **structural breaks** mean that models trained on pre-2022 data may not generalise well to the current market environment. A regime indicator variable has been engineered to account for this.
